# 01a: Data Exploration — Shimizu 20260305 Dataset

**Experiment**: 01 — Averaging order effect on MOLASS decomposition  
**Purpose**: Visual QC of all 6 datasets before running MOLASS  
**Date**: March 2026  
**Data**: `Dropbox\MOLASS\DATA\20260305`

## What we are checking
- UV traces (signal λ and baseline 400 nm) for all 6 datasets
- SAXS intensity traces
- Peak shapes and apparent overlap
- Visual S/N comparison: original vs pre-averaged

## Datasets
| Name | Sample | UV signal λ | Type |
|------|--------|------------|------|
| Apo  | Apo protein | 280 nm | Original |
| Apo2 | Apo protein | 280 nm | Pre-averaged (19 frames) |
| ATP  | ATP-bound   | 290 nm | Original |
| ATP2 | ATP-bound   | 290 nm | Pre-averaged (19 frames) |
| MY   | MY          | 290 nm | Original |
| MY2  | MY          | 290 nm | Pre-averaged (19 frames) |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from molass.DataObjects import SecSaxsData as SSD
from molass.DataObjects.Curve import create_icurve
import molass; assert molass.get_version() >= "0.8.5", \
    f"molass >= 0.8.5 required, found {molass.get_version()}"
print("molass", molass.get_version())

# ── Set your local data root here ──────────────────────────────────────────
DATA_ROOT = r"C:\Users\takahashi\Dropbox\MOLASS\DATA\20260305"
# ───────────────────────────────────────────────────────────────────────────

DATASETS = {
    "Apo":  {"folder": "Apo",  "uv_signal": 280, "pair": "Apo2"},
    "Apo2": {"folder": "Apo2", "uv_signal": 280, "pair": "Apo"},
    "ATP":  {"folder": "ATP",  "uv_signal": 290, "pair": "ATP2"},
    "ATP2": {"folder": "ATP2", "uv_signal": 290, "pair": "ATP"},
    "MY":   {"folder": "MY",   "uv_signal": 290, "pair": "MY2"},
    "MY2":  {"folder": "MY2",  "uv_signal": 290, "pair": "MY"},
}
UV_BASELINE = 400

print("Data root:", DATA_ROOT)
print("Exists:", os.path.isdir(DATA_ROOT))


In [ ]:
# Load all datasets
ssds = {}
for name, info in DATASETS.items():
    folder = os.path.join(DATA_ROOT, info["folder"])
    if os.path.isdir(folder):
        print(f"Loading {name} from {folder} ...")
        ssds[name] = SSD(folder, uv_pickat=info["uv_signal"])
        print(f"  XR shape: {ssds[name].xr.M.shape}  UV shape: {ssds[name].uv.M.shape}")
    else:
        print(f"WARNING: {folder} not found — skipping {name}")

## UV Elution Traces

For each sample pair (original vs pre-averaged), plot:
- UV signal wavelength (280 or 290 nm)
- Baseline wavelength (400 nm)
- Corrected trace = signal − baseline

In [ ]:
SAMPLE_PAIRS = [("Apo", "Apo2", 280), ("ATP", "ATP2", 290), ("MY", "MY2", 290)]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("UV Elution Traces — Original vs Pre-averaged", fontsize=14)

for row, (orig, avg, wv_signal) in enumerate(SAMPLE_PAIRS):
    for col, name in enumerate([orig, avg]):
        ax = axes[row, col]
        if name not in ssds:
            ax.text(0.5, 0.5, f"{name} not loaded", ha="center", va="center", transform=ax.transAxes)
            continue
        ssd = ssds[name]
        c_signal   = create_icurve(None, ssd.uv.M, ssd.uv.wv, wv_signal)
        c_baseline = create_icurve(None, ssd.uv.M, ssd.uv.wv, UV_BASELINE)
        x = np.arange(len(c_signal.y))
        ax.plot(x, c_signal.y,   label=f"{wv_signal} nm", color="steelblue")
        ax.plot(x, c_baseline.y, label=f"{UV_BASELINE} nm (baseline)", color="gray", alpha=0.6)
        ax.plot(x, c_signal.y - c_baseline.y, label="corrected", color="tomato", linewidth=1.2)
        label = "Original" if name == orig else "Pre-averaged (19 frames)"
        ax.set_title(f"{name} — {label}")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Absorbance (AU)")
        ax.legend(fontsize=7)

    # Overlay comparison in 3rd column
    ax = axes[row, 2]
    if orig in ssds and avg in ssds:
        c_orig = create_icurve(None, ssds[orig].uv.M, ssds[orig].uv.wv, wv_signal)
        c_avg  = create_icurve(None, ssds[avg].uv.M,  ssds[avg].uv.wv,  wv_signal)
        cb_orig = create_icurve(None, ssds[orig].uv.M, ssds[orig].uv.wv, UV_BASELINE)
        cb_avg  = create_icurve(None, ssds[avg].uv.M,  ssds[avg].uv.wv,  UV_BASELINE)
        y_orig = c_orig.y - cb_orig.y
        y_avg  = c_avg.y  - cb_avg.y
        # Normalize to peak for shape comparison
        x = np.arange(len(y_orig))
        ax.plot(x, y_orig / y_orig.max(), label="Original (norm)",      color="steelblue")
        xa = np.arange(len(y_avg))
        ax.plot(xa, y_avg  / y_avg.max(),  label="Pre-avg (norm)",       color="tomato", linestyle="--")
        ax.set_title(f"{orig} vs {avg} — corrected UV (normalized)")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Normalized absorbance")
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("01a_uv_traces.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01a_uv_traces.png")

## SAXS Intensity Elution Traces

Plot the integrated SAXS intensity (summed over q, or at low q) as a function of frame number.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("SAXS Intensity Elution Traces — Original vs Pre-averaged", fontsize=14)

for row, (orig, avg, wv_signal) in enumerate(SAMPLE_PAIRS):
    for col, name in enumerate([orig, avg]):
        ax = axes[row, col]
        if name not in ssds:
            ax.text(0.5, 0.5, f"{name} not loaded", ha="center", va="center", transform=ax.transAxes)
            continue
        M = ssds[name].xr.M        # shape: q × frames
        intensity = M.sum(axis=0)  # sum over q → 1D elution trace
        label = "Original" if name == orig else "Pre-averaged (19 frames)"
        ax.plot(intensity, color="steelblue")
        ax.set_title(f"{name} — {label}")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Summed SAXS intensity")

    # Overlay
    ax = axes[row, 2]
    if orig in ssds and avg in ssds:
        I_orig = ssds[orig].xr.M.sum(axis=0)
        I_avg  = ssds[avg].xr.M.sum(axis=0)
        ax.plot(I_orig / I_orig.max(), label="Original (norm)",  color="steelblue")
        ax.plot(I_avg  / I_avg.max(),  label="Pre-avg (norm)",   color="tomato", linestyle="--")
        ax.set_title(f"{orig} vs {avg} — SAXS (normalized)")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Normalized SAXS intensity")
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("01a_saxs_traces.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01a_saxs_traces.png")

## S/N Comparison

Quantify the noise improvement from pre-averaging. 
Method: compute point-to-point variance in the baseline region (before the elution peak).

In [ ]:
print(f"{'Dataset':<8} {'UV peak (AU)':>14} {'UV baseline SD':>16} {'S/N ratio':>12}")
print("-" * 54)

for orig, avg, wv_signal in SAMPLE_PAIRS:
    for name in [orig, avg]:
        if name not in ssds:
            continue
        ssd = ssds[name]
        c_signal   = create_icurve(None, ssd.uv.M, ssd.uv.wv, wv_signal)
        c_baseline = create_icurve(None, ssd.uv.M, ssd.uv.wv, UV_BASELINE)
        corrected = c_signal.y - c_baseline.y
        peak = corrected.max()
        # Use first 20 frames as baseline region (before elution)
        noise = corrected[:20].std()
        snr = peak / noise if noise > 0 else float("inf")
        print(f"{name:<8} {peak:>14.4f} {noise:>16.6f} {snr:>12.1f}")

## Observations

*(Filled in after running cells above — March 6, 2026)*

### Peak shapes
- **Apo/Apo2**: Single sharp, symmetric peak at ~frame 900. Normalized overlay shows near-identical shape.
- **ATP/ATP2**: Main peak at ~frame 950 with a smaller shoulder at ~frame 1200. Shapes match.
- **MY/MY2**: Two clearly resolved peaks (~frame 900 and ~frame 1230). Most complex elution profile. Shapes match.

### S/N comparison
- The UV S/N is **identical** between original and pre-averaged datasets (all pairs: Apo≈215, ATP≈224, MY≈91).
- This likely means the UV data files are unchanged — only the SAXS frames were pre-averaged by Shimizu's tool.
- SAXS baseline noise is visibly lower in the pre-averaged sets (especially clear in MY vs MY2 SAXS traces).

### Unexpected features
- S/N improvement is not visible in UV — need to confirm whether UV data in the `*2` folders is identical to originals (could check file sizes or hash).
- MY has the most complex elution (2 peaks) and lowest S/N, making it the most interesting test case for the averaging-order question.

### Readiness for MOLASS analysis
→ All 6 datasets loaded and visually QC'd. Proceed to `01b_molass_runs.ipynb`.
